In [1]:
import requests
import pandas as pd
import time

In [2]:
BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api/v2/associations"
params = {
    "efo_id": "MONDO_0005180",
    "show_child_traits": "false",
    "size": 100
}
all_associations =[] 
url = BASE_URL
first_request = True

while url:
    if first_request:
        response=requests.get(url,params=params)
        first_request=False
    else:
        response=requests.get(url)

    response.raise_for_status()
    data=response.json()

    page_records = data["_embedded"]["associations"]
    all_associations.extend(page_records)
    next_link = data.get("_links", {}).get("next", {}).get("href")
    url = next_link

    time.sleep(0.2)
    print(f"Fetched {len(all_associations)} associations so far...")

print(f"Done. Total associations collected: {len(all_associations)}")

Fetched 100 associations so far...
Fetched 200 associations so far...
Fetched 300 associations so far...
Fetched 400 associations so far...
Fetched 500 associations so far...
Fetched 600 associations so far...
Fetched 700 associations so far...
Fetched 800 associations so far...
Fetched 814 associations so far...
Done. Total associations collected: 814


In [3]:
CORE_TRAIT_STRINGS = {"Parkinson's disease", "Parkinson disease"}

flat_rows = []

for assoc in all_associations:
    reported = assoc.get("reported_trait", [])
    if not any(t in CORE_TRAIT_STRINGS for t in reported):
        continue

    genes = assoc.get("mapped_genes", [])
    if not genes:
        continue

    p_value = assoc.get("p_value")
    accession_id = assoc.get("accession_id")

    for gene in genes:
        flat_rows.append({
            "gene": gene,
            "p_value": p_value,
            "reported_trait": reported[0] if reported else None,
            "accession_id": accession_id
        })

print(f"Total gene-level rows after filtering: {len(flat_rows)}")

Total gene-level rows after filtering: 710


In [4]:
df = pd.DataFrame(flat_rows)
print(df.shape)
df.head()

(710, 4)


,gene,p_value,reported_trait,accession_id
0,SNCA,3.000000e-12,Parkinson's disease,GCST90475830
1,LRRK2,2.000000e-11,Parkinson's disease,GCST90475830
2,LDLRAD4,2.000000e-11,Parkinson's disease,GCST90479727
3,CDC42BPB,4.000000e-11,Parkinson's disease,GCST90479727
4,LRRK2,3.000000e-12,Parkinson's disease,GCST90479727


In [5]:
df['gene'].nunique()

337

In [6]:
df['p_value'].describe()

count     7.100000e+02
mean      5.309951e-07
std       1.371069e-06
min      4.000000e-170
25%       3.250000e-13
50%       8.500000e-10
75%       2.000000e-07
max       1.000000e-05
Name: p_value, dtype: float64

In [7]:
GENOME_WIDE_SIG = 5e-8
print(f'Original Shape b4 p_value cut-off: {df.shape}')

df_sig=df[df['p_value']<GENOME_WIDE_SIG].copy()
print(f'Shape after p_value cut-off: {df_sig.shape}')

df_sig_dedupe=df_sig.sort_values('p_value').drop_duplicates(subset='gene',keep='first')
print(f'Shape after dedupe: {df_sig_dedupe.shape}')

df_sig_dedupe.head()

Original Shape b4 p_value cut-off: (710, 4)
Shape after p_value cut-off: (474, 4)
Shape after dedupe: (208, 4)


,gene,p_value,reported_trait,accession_id
154,SNCA,4.000000e-170,Parkinson's disease,GCST90308590
104,GBA1,3.000000e-90,Parkinson disease,GCST90428733
257,HMGN2P18,1.000000e-75,Parkinson's disease,GCST90308590
159,TMEM175,5.000000e-75,Parkinson's disease,GCST90308590
189,LINC02210-CRHR1,1.000000e-69,Parkinson's disease,GCST90308590


In [9]:
import sys
!{sys.executable} -m pip install abagen

  Using cached abagen-0.1.3-py3-none-any.whl.metadata (8.6 kB)
  Using cached nibabel-5.4.2-py3-none-any.whl.metadata (8.9 kB)
Using cached abagen-0.1.3-py3-none-any.whl (3.7 MB)
Using cached nibabel-5.4.2-py3-none-any.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [abagen]]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import abagen 

atlas = abagen.fetch_desikan_killiany()
atlas_image=atlas['image']
atlas_info=atlas['info']

print(atlas_image)
print(atlas_info)

/Users/sannidhyabiswas/Documents/Dev-Workspace/ML/.venv/lib/python3.13/site-packages/abagen/mouse/io.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


/Users/sannidhyabiswas/Documents/Dev-Workspace/ML/.venv/lib/python3.13/site-packages/abagen/data/atlas-desikankilliany.nii.gz
/Users/sannidhyabiswas/Documents/Dev-Workspace/ML/.venv/lib/python3.13/site-packages/abagen/data/atlas-desikankilliany.csv


In [13]:
donor_ids = ['9861', '10021', '12876', '14380', '15496', '15697']

for donor in donor_ids:
    try:
        abagen.fetch_microarray(donors=[donor], resume=True)
        print(f"Donor {donor} done")
    except Exception as e:
        print(f"Donor {donor} failed: {e}")

Donor 9861 done
Donor 10021 done
Donor 12876 done
Donor 14380 done
Donor 15496 done
Donor 15697 done
